# Searchlight Results: Choice-Frequency Classification (±1)

Whole-brain searchlight binary classification of the choice-frequency label
(`first_stim_frequ` = ±1) from GLMsingle single-trial betas on the non-figure
stimulus subset (6 stimuli, 3 per frequency class).

Produced by `run_frequency_searchlight.py`. Feature preprocessing: run-demeaning only
(per-voxel run-mean subtracted from ALL trials before dropping freq=0). No category
demeaning needed — frequency is perfectly balanced across categories on the non-figure
subset by design.

Chance level = 0.5 (balanced binary).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from scipy import stats
from statsmodels.stats.multitest import multipletests
import nibabel as nib
from nilearn import image, plotting
from nilearn.image import resample_to_img, math_img, index_img

DERIV_DIR = Path("/Users/hugofluhr/phd_local/data/LearningHabits/dev_sample/bids_dataset/derivatives")
SL_DIR = DERIV_DIR / "searchlight"
MASK_DIR = Path("/Users/hugofluhr/phd_local/data/LearningHabits/dev_sample/masks/MNI152NLin2009cAsym")

MASKS = {
    "visualcortex": DERIV_DIR / "decoding" / "visual_cortex_mask.nii.gz",
    "fusiform": MASK_DIR / "fusiform_mask_MNI152NLin2009cAsym.nii",
    "vmpfc": MASK_DIR / "vmpfc_bartra2013_MNI152NLin2009cAsym.nii",
    "striatum": MASK_DIR / "striatum_bartra2013_MNI152NLin2009cAsym.nii",
    "habit": MASK_DIR / "habit_Guida2022_MNI152NLin2009cAsym.nii",
    "putamen": MASK_DIR / "putamen_AAL_MNI152NLin2009cAsym.nii",
    "premotor": MASK_DIR / "premotor_HMAT_MNI152NLin2009cAsym.nii",
    "parietal": MASK_DIR / "parietal_AAL_MNI152NLin2009cAsym.nii",
}
MASK_LABELS = {
    "visualcortex": "Visual cortex", "fusiform": "Fusiform",
    "vmpfc": "vmPFC", "striatum": "Striatum",
    "habit": "Habit (Guida)", "putamen": "Putamen",
    "premotor": "Premotor", "parietal": "Parietal",
}

CHANCE = 0.5

## 1. Data Loading & Completeness Check

In [ ]:
# Discover per-subject searchlight maps
map_paths = {}
for p in sorted(SL_DIR.glob("sub-*/sub-*_searchlight_frequency.nii.gz")):
    sid = p.parent.name.replace("sub-", "")
    map_paths[sid] = p

print(f"Found {len(map_paths)} subjects: {', '.join(sorted(map_paths.keys())[:10])}...")

# Load into a 4D image
imgs = [nib.load(str(map_paths[s])) for s in sorted(map_paths)]
imgs_4d = image.concat_imgs(imgs)
print(f"4D shape: {imgs_4d.shape}")
subjects = sorted(map_paths.keys())

## 2. Group Mean Accuracy Map

In [ ]:
mean_img = image.mean_img(imgs_4d)

fig, axes = plt.subplots(1, 1, figsize=(14, 5))
display = plotting.plot_stat_map(
    mean_img, threshold=CHANCE,
    title="Group mean accuracy (frequency ±1 classification)",
    cut_coords=(-2, -12, 4), display_mode="ortho",
    cmap="RdYlGn", vmax=0.65,
    figure=fig,
)
plt.tight_layout()
plt.show()

# Also show axial slices through visual cortex / temporal / frontal
fig2, axes2 = plt.subplots(1, 1, figsize=(14, 4))
plotting.plot_stat_map(
    mean_img, threshold=CHANCE,
    title="Group mean accuracy — axial slices",
    display_mode="z", cut_coords=[-20, -10, 0, 10, 20, 40, 55],
    cmap="RdYlGn", vmax=0.65,
    figure=fig2,
)
plt.tight_layout()
plt.show()

## 3. Group Inference: Voxelwise t-test vs. Chance (0.5)

One-sample t-test at each voxel against chance (0.5), followed by FDR correction
(Benjamini-Hochberg, q < 0.05).

In [ ]:
# Extract subjects × voxels array (mask to brain)
data_4d = imgs_4d.get_fdata()                     # (x, y, z, n_subjects)
brain_mask = np.all(data_4d > 0, axis=3)          # voxels present in all subjects
n_brain = brain_mask.sum()
print(f"Brain voxels present in all subjects: {n_brain:,}")

vals = data_4d[brain_mask]                         # (n_voxels, n_subjects)
t_vals, p_vals = stats.ttest_1samp(vals, CHANCE, axis=1)

# FDR correction
reject, p_fdr, _, _ = multipletests(p_vals, alpha=0.05, method='fdr_bh')
n_sig = reject.sum()
print(f"Significant voxels (FDR q<0.05): {n_sig:,} / {n_brain:,} ({100*n_sig/n_brain:.1f}%)")

# Separate above-chance vs below-chance
above_chance = reject & (t_vals > 0)
below_chance = reject & (t_vals < 0)
print(f"  Above chance: {above_chance.sum():,}")
print(f"  Below chance: {below_chance.sum():,}")

# Build t-stat map and FDR-thresholded map
t_map = np.zeros(brain_mask.shape)
t_map[brain_mask] = t_vals
t_img = nib.Nifti1Image(t_map, imgs_4d.affine, imgs_4d.header)

# FDR-thresholded t-map (keep only significant voxels)
t_fdr = np.zeros(brain_mask.shape)
t_fdr[brain_mask] = np.where(reject, t_vals, 0)
t_fdr_img = nib.Nifti1Image(t_fdr, imgs_4d.affine, imgs_4d.header)

In [ ]:
# Plot FDR-thresholded t-map
fig, ax = plt.subplots(1, 1, figsize=(14, 5))
plotting.plot_stat_map(
    t_fdr_img, threshold=0.01,  # already zeroed non-significant voxels
    title=f"Frequency classification: FDR-corrected t-map (q<0.05, {n_sig:,} voxels)",
    cut_coords=(-2, -12, 4), display_mode="ortho",
    cmap="cold_hot",
    figure=fig,
)
plt.tight_layout()
plt.show()

# Axial slices
fig2, ax2 = plt.subplots(1, 1, figsize=(14, 4))
plotting.plot_stat_map(
    t_fdr_img, threshold=0.01,
    title="FDR-corrected t-map — axial slices",
    display_mode="z", cut_coords=[-20, -10, 0, 10, 20, 40, 55],
    cmap="cold_hot",
    figure=fig2,
)
plt.tight_layout()
plt.show()

## 4. Peak Coordinates

Extract local maxima from the FDR-thresholded t-map and report their MNI coordinates
and anatomical labels (Harvard-Oxford atlas).

In [ ]:
from nilearn.reporting import get_clusters_table

# Use unthresholded t-map with a statistical threshold for cluster extraction
# threshold at the FDR-corrected level
if n_sig > 0:
    # Find the minimum |t| among FDR-significant voxels as the threshold
    sig_t = np.abs(t_vals[reject])
    t_thresh = sig_t.min()
    print(f"FDR threshold corresponds to |t| >= {t_thresh:.2f}")

    clusters_table = get_clusters_table(
        t_img, stat_threshold=t_thresh, min_distance=12,
        cluster_threshold=10,
    )
    print(f"\n{len(clusters_table)} peaks found:")
    display(clusters_table.head(20))
else:
    print("No FDR-significant voxels — skipping peak extraction.")

## 5. ROI Mean Accuracy

Mean searchlight accuracy within each RSA ROI mask — bridges the searchlight and
ROI-level analyses. Compares with RSA frequency β from `rsa_roi_results.ipynb`.

In [ ]:
# Load and resample ROI masks to searchlight space
ref_img = imgs[0]

def load_roi_mask(mask_path, ref_img):
    """Load and resample an ROI mask, handling 4D singletons."""
    roi_mni = nib.load(str(mask_path))
    roi_func = resample_to_img(roi_mni, ref_img, interpolation='nearest')
    if roi_func.ndim == 4 and roi_func.shape[3] == 1:
        roi_func = index_img(roi_func, 0)
    return roi_func.get_fdata() > 0

roi_masks_arr = {}
for name, path in MASKS.items():
    if path.exists():
        roi_masks_arr[name] = load_roi_mask(path, ref_img)
    else:
        print(f"WARNING: mask not found: {name} -> {path}")

# Compute per-subject mean accuracy within each ROI
summary_rows = []
for name, mask_arr in roi_masks_arr.items():
    n_vox = mask_arr.sum()
    for i, sid in enumerate(subjects):
        subj_data = data_4d[:, :, :, i]
        roi_vals = subj_data[mask_arr & brain_mask]
        mean_acc = roi_vals.mean() if len(roi_vals) > 0 else np.nan
        summary_rows.append({"subject": sid, "roi": name, "mean_accuracy": mean_acc, "n_voxels": len(roi_vals)})

summary_df = pd.DataFrame(summary_rows)

# Group-level stats per ROI
roi_stats = []
for name in roi_masks_arr:
    accs = summary_df.loc[summary_df["roi"] == name, "mean_accuracy"].values
    t, p = stats.ttest_1samp(accs, CHANCE)
    roi_stats.append({
        "ROI": MASK_LABELS.get(name, name),
        "n_voxels": roi_masks_arr[name].sum(),
        "mean_acc": accs.mean(),
        "std": accs.std(),
        "t": t, "p": p, "n": len(accs),
    })

roi_stats_df = pd.DataFrame(roi_stats)
# FDR across ROIs
_, roi_stats_df["p_fdr"], _, _ = multipletests(roi_stats_df["p"], alpha=0.05, method="fdr_bh")
roi_stats_df["sig"] = roi_stats_df["p_fdr"].apply(lambda p: "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else "")

display(roi_stats_df.round(4))

In [ ]:
# Bar chart: mean searchlight accuracy per ROI
fig, ax = plt.subplots(figsize=(10, 5))
roi_order = list(roi_masks_arr.keys())
roi_labels = [MASK_LABELS.get(r, r) for r in roi_order]
means = [roi_stats_df.loc[roi_stats_df["ROI"] == MASK_LABELS.get(r, r), "mean_acc"].values[0] for r in roi_order]
sems = [summary_df.loc[summary_df["roi"] == r, "mean_accuracy"].std() / np.sqrt(len(subjects)) for r in roi_order]
sigs = [roi_stats_df.loc[roi_stats_df["ROI"] == MASK_LABELS.get(r, r), "sig"].values[0] for r in roi_order]

colors = ["#4C72B0" if m > CHANCE else "#C44E52" for m in means]
bars = ax.bar(roi_labels, means, yerr=sems, capsize=4, color=colors, edgecolor="white", linewidth=0.5)
ax.axhline(CHANCE, color="grey", ls="--", lw=1, label="chance (0.5)")
for i, (bar, sig) in enumerate(zip(bars, sigs)):
    if sig:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + sems[i] + 0.003,
                sig, ha="center", va="bottom", fontsize=11)

ax.set_ylabel("Mean searchlight accuracy")
ax.set_title("Choice-frequency classification: mean searchlight accuracy per ROI (n=58)")
ax.set_ylim(0.46, max(means) + 0.04)
ax.legend(loc="lower right")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

## 6. Comparison with Category Searchlight

Overlay the frequency searchlight with the existing category searchlight to see
whether the frequency signal occupies the same visual cortex territory or extends
beyond it.

In [ ]:
# Load category searchlight maps if available
cat_sl_paths = sorted(SL_DIR.glob("sub-*/sub-*_searchlight.nii.gz"))  # original category searchlight
if cat_sl_paths:
    cat_imgs = [nib.load(str(p)) for p in cat_sl_paths]
    cat_4d = image.concat_imgs(cat_imgs)
    cat_mean = image.mean_img(cat_4d)

    fig, axes = plt.subplots(2, 1, figsize=(14, 8))
    plotting.plot_stat_map(
        mean_img, threshold=CHANCE,
        title="Frequency searchlight (group mean)",
        display_mode="z", cut_coords=[-15, -5, 5, 15, 30, 50],
        cmap="RdYlGn", vmax=0.65,
        axes=axes[0],
    )
    plotting.plot_stat_map(
        cat_mean, threshold=CHANCE,
        title="Category searchlight (group mean)",
        display_mode="z", cut_coords=[-15, -5, 5, 15, 30, 50],
        cmap="RdYlGn", vmax=0.65,
        axes=axes[1],
    )
    plt.tight_layout()
    plt.show()

    # Correlation of group mean accuracy maps
    freq_data = mean_img.get_fdata()[brain_mask]
    cat_data = cat_mean.get_fdata()[brain_mask]
    r, p_corr = stats.pearsonr(freq_data, cat_data)
    print(f"Spatial correlation (group mean maps): r = {r:.3f}, p = {p_corr:.2e}")
else:
    print("No category searchlight maps found — skipping comparison.")

## 7. Individual Subject Maps (Spot-check)

Show a few representative subjects to verify the effect is consistent and not driven
by outliers.

In [ ]:
# Pick subjects by visual cortex mean accuracy: high, median, low
vc_accs = summary_df.loc[summary_df["roi"] == "visualcortex"].set_index("subject")["mean_accuracy"]
ranked = vc_accs.sort_values()
pick_subs = [ranked.index[-1], ranked.index[len(ranked)//2], ranked.index[0]]

fig, axes = plt.subplots(len(pick_subs), 1, figsize=(14, 4 * len(pick_subs)))
for ax, sid in zip(axes, pick_subs):
    subj_img = nib.load(str(map_paths[sid]))
    plotting.plot_stat_map(
        subj_img, threshold=CHANCE,
        title=f"sub-{sid} (VC mean acc = {vc_accs[sid]:.3f})",
        display_mode="z", cut_coords=[-15, -5, 5, 15, 30],
        cmap="RdYlGn", vmax=0.7,
        axes=ax,
    )
plt.tight_layout()
plt.show()

## 8. Findings Summary

*Populated after running the notebook.*